In [ ]:
# ============================================================
# STEP 1 — Caricamento, pulizia e preparazione EU-LS 2022
# Does Social Media Use Reduce Loneliness?
# ============================================================

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

os.makedirs('output/dataset', exist_ok=True)
os.makedirs('output/figures', exist_ok=True)

# ------------------------------------------------------------------
# 1.1  CARICAMENTO
# ------------------------------------------------------------------
df_raw = pd.read_csv('eu_loneliness_survey_eu27_values.csv', low_memory=False)
print(f"Dataset grezzo: {df_raw.shape[0]} righe × {df_raw.shape[1]} colonne")

# ------------------------------------------------------------------
# 1.2  SELEZIONE COLONNE DI INTERESSE
# ------------------------------------------------------------------
COLS = ['loneliness_ucla_a', 'loneliness_ucla_b', 'loneliness_ucla_c',
        'sm_time_a', 'country', 'age', 'gender', 'education', 'income_decile']

df = df_raw[COLS].copy()

# ------------------------------------------------------------------
# 1.3  COSTRUZIONE score_UCLA (variabile dipendente)
# ------------------------------------------------------------------
# Scala UCLA 3 item, valori 1–3 (999 = non risposta)
# Score = somma dei 3 item → range 3–9 (9 = massima solitudine)

for col in ['loneliness_ucla_a', 'loneliness_ucla_b', 'loneliness_ucla_c']:
    df[col] = df[col].replace(999, np.nan)

df['score_UCLA'] = df[['loneliness_ucla_a',
                        'loneliness_ucla_b',
                        'loneliness_ucla_c']].sum(axis=1, min_count=3)

print("\n--- score_UCLA ---")
print(df['score_UCLA'].describe().round(2))
print(f"Missing: {df['score_UCLA'].isna().sum()} ({df['score_UCLA'].isna().mean()*100:.1f}%)")

# ------------------------------------------------------------------
# 1.4  COSTRUZIONE intensita_sm (variabile indipendente — ORDINALE)
# ------------------------------------------------------------------
# sm_time_a: 1–8 = livelli crescenti di tempo sui social media
#            999  = non usa social media → ricodificato come 0
# Variabile finale: 0 (non utente) + 1–8 (utente per intensità)
# Range finale: 0–8 (0 = mai, 8 = uso massimo)

df['intensita_sm'] = df['sm_time_a'].replace(999, 0)

print("\n--- intensita_sm ---")
print(df['intensita_sm'].value_counts().sort_index())
print(f"\nMedia: {df['intensita_sm'].mean():.2f}")
print(f"Missing: {df['intensita_sm'].isna().sum()}")

# ------------------------------------------------------------------
# 1.5  COSTRUZIONE fascia_eta (variabile moderatrice)
# ------------------------------------------------------------------
bins   = [15, 34, 54, 74, 120]
labels = ['16–34', '35–54', '55–74', '75+']

df['fascia_eta'] = pd.cut(df['age'], bins=bins, labels=labels, right=True)

print("\n--- Distribuzione fasce d'età ---")
print(df['fascia_eta'].value_counts().sort_index())

# ------------------------------------------------------------------
# 1.6  PULIZIA VARIABILI DI CONTROLLO
# ------------------------------------------------------------------
for col in ['gender', 'education', 'income_decile']:
    df[col] = df[col].replace(999, np.nan)

print("\n--- Missing variabili di controllo ---")
for col in ['gender', 'education', 'income_decile']:
    n = df[col].isna().sum()
    print(f"  {col:20s}: {n} ({n/len(df)*100:.1f}%)")

# ------------------------------------------------------------------
# 1.7  LISTWISE DELETION
# ------------------------------------------------------------------
COLONNE_MODELLO = ['score_UCLA', 'intensita_sm', 'age', 'fascia_eta',
                   'gender', 'education', 'income_decile', 'country']

n_prima = len(df)
df_clean = df.dropna(subset=COLONNE_MODELLO).copy()
n_dopo   = len(df_clean)

print(f"\n--- Listwise deletion ---")
print(f"Righe prima : {n_prima}")
print(f"Righe dopo  : {n_dopo}")
print(f"Rimosse     : {n_prima - n_dopo} ({(n_prima - n_dopo)/n_prima*100:.1f}%)")

# ------------------------------------------------------------------
# 1.8  DATAFRAME FINALE
# ------------------------------------------------------------------
COLONNE_FINALI = ['country', 'intensita_sm', 'score_UCLA',
                  'fascia_eta', 'age', 'gender', 'education', 'income_decile']

df_final = df_clean[COLONNE_FINALI].reset_index(drop=True)
df_final.columns = ['paese', 'intensita_sm', 'score_UCLA',
                    'fascia_eta', 'eta', 'sesso', 'education', 'income']

print("\n=== DataFrame finale ===")
print(f"Shape: {df_final.shape}")
print(f"Paesi presenti: {df_final['paese'].nunique()}")
print(f"\nAnteprima:\n{df_final.head(5)}")

# ------------------------------------------------------------------
# 1.9  GRAFICI DIAGNOSTICI
# ------------------------------------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Distribuzione score UCLA
axes[0].hist(df_final['score_UCLA'], bins=7, color='steelblue', edgecolor='white')
axes[0].set_title('Distribuzione score UCLA')
axes[0].set_xlabel('Score (3–9)')
axes[0].set_ylabel('Frequenza')

# Distribuzione intensita_sm (ordinale 0–8)
conteggi = df_final['intensita_sm'].value_counts().sort_index()
axes[1].bar(conteggi.index, conteggi.values, color='steelblue', edgecolor='white')
axes[1].set_title('Distribuzione intensità uso social media')
axes[1].set_xlabel('Livello (0 = non utente, 8 = uso massimo)')
axes[1].set_ylabel('Frequenza')
axes[1].set_xticks(range(9))

# Score UCLA medio per livello di intensita_sm
medie_globali = df_final.groupby('intensita_sm')['score_UCLA'].mean()
axes[2].plot(medie_globali.index, medie_globali.values,
             marker='o', color='steelblue', linewidth=2)
axes[2].set_title('Score UCLA medio per intensità social media')
axes[2].set_xlabel('Livello intensita_sm')
axes[2].set_ylabel('Score UCLA medio')
axes[2].set_xticks(range(9))

plt.tight_layout()
fig.savefig('output/figures/step1_diagnostici.png', dpi=150, bbox_inches='tight')
plt.show()

# ------------------------------------------------------------------
# 1.10  SALVATAGGIO
# ------------------------------------------------------------------
df_final.to_csv('output/dataset/eu_ls_clean.csv', index=False)
print("\nSalvato: output/dataset/eu_ls_clean.csv")
print("Salvato: output/figures/step1_diagnostici.png")

In [ ]:
# ============================================================
# STEP 2 — Analisi esplorativa (EDA)
# Does Social Media Use Reduce Loneliness?
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

pd.set_option('display.float_format', '{:.2f}'.format)

# ------------------------------------------------------------------
# 2.1  CARICAMENTO
# ------------------------------------------------------------------
df = pd.read_csv('output/dataset/eu_ls_clean.csv')
df['fascia_eta'] = pd.Categorical(df['fascia_eta'],
                                   categories=['16–34', '35–54', '55–74', '75+'],
                                   ordered=True)
print(f"Dataset: {df.shape[0]} righe × {df.shape[1]} colonne")

# ------------------------------------------------------------------
# 2.2  STATISTICHE DESCRITTIVE PER FASCIA D'ETÀ
# ------------------------------------------------------------------
desc = df.groupby('fascia_eta', observed=True).agg(
    n               = ('score_UCLA', 'count'),
    ucla_media      = ('score_UCLA', 'mean'),
    ucla_std        = ('score_UCLA', 'std'),
    sm_media        = ('intensita_sm', 'mean'),
    sm_std          = ('intensita_sm', 'std'),
).round(2)

print("\n=== Statistiche per fascia d'età ===")
print(desc)

# ------------------------------------------------------------------
# 2.3  STATISTICHE DESCRITTIVE PER PAESE
# ------------------------------------------------------------------
desc_paese = df.groupby('paese', observed=True).agg(
    n           = ('score_UCLA', 'count'),
    ucla_media  = ('score_UCLA', 'mean'),
    sm_media    = ('intensita_sm', 'mean'),
).round(2).sort_values('ucla_media', ascending=False)

print("\n=== Statistiche per paese (ordinato per UCLA medio) ===")
print(desc_paese)

# ------------------------------------------------------------------
# 2.4  CORRELAZIONE BIVARIATA — tutto il campione
# ------------------------------------------------------------------
pearson_r, pearson_p   = stats.pearsonr(df['intensita_sm'], df['score_UCLA'])
spearman_r, spearman_p = stats.spearmanr(df['intensita_sm'], df['score_UCLA'])

print("\n=== Correlazione bivariata (tutto il campione) ===")
print(f"Pearson  r = {pearson_r:.3f}  p = {pearson_p:.4f}")
print(f"Spearman r = {spearman_r:.3f}  p = {spearman_p:.4f}")

# ------------------------------------------------------------------
# 2.5  CORRELAZIONE PER FASCIA D'ETÀ
# ------------------------------------------------------------------
print("\n=== Correlazione Pearson per fascia d'età ===")
corr_per_fascia = []
for fascia in ['16–34', '35–54', '55–74', '75+']:
    sub = df[df['fascia_eta'] == fascia]
    r, p = stats.pearsonr(sub['intensita_sm'], sub['score_UCLA'])
    corr_per_fascia.append({'fascia': fascia, 'n': len(sub),
                            'pearson_r': round(r, 3), 'p_value': round(p, 4)})
    print(f"  {fascia:6s}  r = {r:.3f}  p = {p:.4f}  (n={len(sub)})")

# ------------------------------------------------------------------
# 2.6  GRAFICI EDA
# ------------------------------------------------------------------
fig = plt.figure(figsize=(16, 12))

# --- Fig A: Heatmap UCLA medio per paese × fascia d'età ---
ax1 = fig.add_subplot(2, 2, 1)
pivot = df.groupby(['paese', 'fascia_eta'],
                    observed=True)['score_UCLA'].mean().unstack()
sns.heatmap(pivot, ax=ax1, cmap='YlOrRd', annot=True, fmt='.1f',
            linewidths=0.3, cbar_kws={'label': 'Score UCLA medio'})
ax1.set_title('Score UCLA medio per paese × fascia d\'età')
ax1.set_xlabel('Fascia d\'età')
ax1.set_ylabel('Paese (codice numerico)')

# --- Fig B: Line plot UCLA medio per livello intensita_sm × fascia d'età ---
ax2 = fig.add_subplot(2, 2, 2)
colori = {'16–34': '#4C72B0', '35–54': '#DD8452',
          '55–74': '#55A868', '75+': '#C44E52'}
for fascia in ['16–34', '35–54', '55–74', '75+']:
    sub = df[df['fascia_eta'] == fascia]
    medie = sub.groupby('intensita_sm')['score_UCLA'].mean()
    ax2.plot(medie.index, medie.values, marker='o', linewidth=2,
             label=fascia, color=colori[fascia])
ax2.set_title('Score UCLA medio per intensità SM × fascia d\'età')
ax2.set_xlabel('Livello intensita_sm (0 = non utente, 8 = uso massimo)')
ax2.set_ylabel('Score UCLA medio')
ax2.set_xticks(range(9))
ax2.legend(title='Fascia d\'età')
ax2.axhline(df['score_UCLA'].mean(), color='gray',
            linestyle='--', linewidth=0.8, label='Media globale')

# --- Fig C: Boxplot score UCLA per fascia d'età ---
ax3 = fig.add_subplot(2, 2, 3)
df.boxplot(column='score_UCLA', by='fascia_eta', ax=ax3,
           boxprops=dict(color='steelblue'),
           medianprops=dict(color='red', linewidth=2),
           whiskerprops=dict(color='steelblue'),
           capprops=dict(color='steelblue'),
           flierprops=dict(marker='o', markersize=2, alpha=0.3))
ax3.set_title('Distribuzione score UCLA per fascia d\'età')
ax3.set_xlabel('Fascia d\'età')
ax3.set_ylabel('Score UCLA')
plt.suptitle('')

# --- Fig D: Boxplot score UCLA per livello intensita_sm ---
ax4 = fig.add_subplot(2, 2, 4)
df.boxplot(column='score_UCLA', by='intensita_sm', ax=ax4,
           boxprops=dict(color='steelblue'),
           medianprops=dict(color='red', linewidth=2),
           whiskerprops=dict(color='steelblue'),
           capprops=dict(color='steelblue'),
           flierprops=dict(marker='o', markersize=2, alpha=0.3))
ax4.set_title('Distribuzione score UCLA per livello intensita_sm')
ax4.set_xlabel('Livello intensita_sm')
ax4.set_ylabel('Score UCLA')
plt.suptitle('')

plt.tight_layout()
fig.savefig('output/figures/step2_eda.png', dpi=150, bbox_inches='tight')
plt.show()

# ------------------------------------------------------------------
# 2.7  RIEPILOGO RISULTATI EDA
# ------------------------------------------------------------------
print("\n=== Riepilogo EDA ===")
print(f"Score UCLA medio globale       : {df['score_UCLA'].mean():.2f}")
print(f"Intensita_sm media globale     : {df['intensita_sm'].mean():.2f}")
print(f"Correlazione Pearson globale   : r = {pearson_r:.3f} (p = {pearson_p:.4f})")
print(f"Correlazione Spearman globale  : r = {spearman_r:.3f} (p = {spearman_p:.4f})")
print("\nFascia con UCLA più alto       :", desc['ucla_media'].idxmax())
print("Fascia con UCLA più basso      :", desc['ucla_media'].idxmin())
print("Fascia con SM più intenso      :", desc['sm_media'].idxmax())
print("Fascia con SM meno intenso     :", desc['sm_media'].idxmin())

In [ ]:
# ============================================================
# STEP 3 — Regressione OLS individuale
# Does Social Media Use Reduce Loneliness?
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import statsmodels.formula.api as smf
from scipy import stats

pd.set_option('display.float_format', '{:.4f}'.format)

# ------------------------------------------------------------------
# 3.1  CARICAMENTO
# ------------------------------------------------------------------
df = pd.read_csv('output/dataset/eu_ls_clean.csv')
df['fascia_eta'] = pd.Categorical(df['fascia_eta'],
                                   categories=['16–34', '35–54', '55–74', '75+'],
                                   ordered=True)

# Termine quadratico
df['intensita_sm2'] = df['intensita_sm'] ** 2

# Dummies paese (paese 1 = riferimento, escluso automaticamente da C())
print(f"Dataset: {df.shape[0]} righe · {df['paese'].nunique()} paesi")

# ------------------------------------------------------------------
# 3.2  MODELLO OLS — TUTTO IL CAMPIONE
# ------------------------------------------------------------------
formula_full = ('score_UCLA ~ intensita_sm + intensita_sm2 '
                '+ sesso + education + income + C(paese)')

model_full  = smf.ols(formula_full, data=df).fit(cov_type='HC3')

print("\n=== OLS — tutto il campione ===")
print(f"N = {int(model_full.nobs)}  |  R² = {model_full.rsquared:.4f}"
      f"  |  R² adj = {model_full.rsquared_adj:.4f}")
print(f"\nCoefficienti chiave:")
chiavi = ['Intercept', 'intensita_sm', 'intensita_sm2',
          'sesso', 'education', 'income']
for k in chiavi:
    b  = model_full.params[k]
    se = model_full.bse[k]
    p  = model_full.pvalues[k]
    ci_lo, ci_hi = model_full.conf_int().loc[k]
    sig = '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else ''))
    print(f"  {k:20s}  β={b:7.4f}  SE={se:.4f}  p={p:.4f}  "
          f"CI=[{ci_lo:.4f}, {ci_hi:.4f}]  {sig}")

# ------------------------------------------------------------------
# 3.3  MODELLO OLS — SEPARATO PER FASCIA D'ETÀ
# ------------------------------------------------------------------
fasce     = ['16–34', '35–54', '55–74', '75+']
risultati = {}

print("\n=== OLS — per fascia d'età ===")
for fascia in fasce:
    sub = df[df['fascia_eta'] == fascia].copy()
    mod = smf.ols(formula_full, data=sub).fit(cov_type='HC3')
    risultati[fascia] = mod

    b1  = mod.params['intensita_sm']
    b2  = mod.params['intensita_sm2']
    p1  = mod.pvalues['intensita_sm']
    p2  = mod.pvalues['intensita_sm2']
    r2  = mod.rsquared

    sig1 = '***' if p1<0.001 else ('**' if p1<0.01 else ('*' if p1<0.05 else 'n.s.'))
    sig2 = '***' if p2<0.001 else ('**' if p2<0.01 else ('*' if p2<0.05 else 'n.s.'))

    print(f"\n  Fascia {fascia}  (N={int(mod.nobs)}, R²={r2:.4f})")
    print(f"    intensita_sm   β={b1:.4f}  p={p1:.4f} {sig1}")
    print(f"    intensita_sm²  β={b2:.4f}  p={p2:.4f} {sig2}")

# ------------------------------------------------------------------
# 3.4  TABELLA COMPARATIVA β PER FASCIA
# ------------------------------------------------------------------
righe = []
for fascia in fasce:
    mod = risultati[fascia]
    for var in ['intensita_sm', 'intensita_sm2']:
        righe.append({
            'fascia'  : fascia,
            'variabile': var,
            'beta'    : round(mod.params[var], 4),
            'SE'      : round(mod.bse[var], 4),
            'p_value' : round(mod.pvalues[var], 4),
            'CI_low'  : round(mod.conf_int().loc[var, 0], 4),
            'CI_high' : round(mod.conf_int().loc[var, 1], 4),
            'sig'     : ('***' if mod.pvalues[var]<0.001
                         else ('**' if mod.pvalues[var]<0.01
                         else ('*' if mod.pvalues[var]<0.05 else 'n.s.')))
        })

tab = pd.DataFrame(righe)
print("\n=== Tabella comparativa β × fascia d'età ===")
print(tab.to_string(index=False))
tab.to_csv('output/dataset/step3_coefficienti.csv', index=False)
print("\nSalvato: output/dataset/step3_coefficienti.csv")

# ------------------------------------------------------------------
# 3.5  VERIFICA H₀ / H₁
# ------------------------------------------------------------------
print("\n=== Verifica ipotesi ===")
print("\nH₀: intensita_sm non associata a score_UCLA in nessuna fascia")
print("H₁: associazione presente e variabile per fascia d'età\n")

for fascia in fasce:
    mod  = risultati[fascia]
    p1   = mod.pvalues['intensita_sm']
    p2   = mod.pvalues['intensita_sm2']
    sig  = (p1 < 0.05) or (p2 < 0.05)
    esito = "→ associazione PRESENTE (contro H₀)" if sig else "→ associazione ASSENTE (a favore H₀)"
    print(f"  {fascia:6s}  intensita_sm p={p1:.4f} | intensita_sm² p={p2:.4f}  {esito}")

# ------------------------------------------------------------------
# 3.6  GRAFICI
# ------------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Fig A: β₁ e β₂ per fascia con intervalli di confidenza ---
x      = np.arange(len(fasce))
width  = 0.35
colori = ['#4C72B0', '#DD8452']

for i, var in enumerate(['intensita_sm', 'intensita_sm2']):
    betas  = [risultati[f].params[var] for f in fasce]
    ci_lo  = [risultati[f].conf_int().loc[var, 0] for f in fasce]
    ci_hi  = [risultati[f].conf_int().loc[var, 1] for f in fasce]
    yerr   = [[b - lo for b, lo in zip(betas, ci_lo)],
               [hi - b for b, hi in zip(betas, ci_hi)]]
    label  = 'β₁ (intensita_sm)' if i == 0 else 'β₂ (intensita_sm²)'
    axes[0].bar(x + i*width, betas, width, yerr=yerr, capsize=4,
                color=colori[i], alpha=0.8, label=label)

axes[0].axhline(0, color='black', linewidth=0.8, linestyle='--')
axes[0].set_xticks(x + width/2)
axes[0].set_xticklabels(fasce)
axes[0].set_title('Coefficienti OLS β₁ e β₂ per fascia d\'età')
axes[0].set_xlabel('Fascia d\'età')
axes[0].set_ylabel('Valore β (con IC 95%)')
axes[0].legend()

# --- Fig B: curve fitted per fascia ---
sm_range = np.linspace(0, 8, 100)
for fascia in fasce:
    mod = risultati[fascia]
    b0  = mod.params['Intercept']
    b1  = mod.params['intensita_sm']
    b2  = mod.params['intensita_sm2']
    # media dei controlli per fascia
    sub = df[df['fascia_eta'] == fascia]
    adj = (mod.params.get('sesso', 0)     * sub['sesso'].mean() +
           mod.params.get('education', 0) * sub['education'].mean() +
           mod.params.get('income', 0)    * sub['income'].mean())
    y_pred = b0 + b1*sm_range + b2*sm_range**2 + adj
    axes[1].plot(sm_range, y_pred, linewidth=2, label=fascia)

axes[1].axhline(df['score_UCLA'].mean(), color='gray',
                linestyle='--', linewidth=0.8)
axes[1].set_title('Curve OLS fitted per fascia d\'età')
axes[1].set_xlabel('Livello intensita_sm')
axes[1].set_ylabel('Score UCLA predetto')
axes[1].set_xticks(range(9))
axes[1].legend(title='Fascia d\'età')

plt.tight_layout()
fig.savefig('output/figures/step3_ols.png', dpi=150, bbox_inches='tight')
plt.show()
print("Salvato: output/figures/step3_ols.png")

In [ ]:
# ============================================================
# STEP 4 — Confronto modelli annidati + R² parziale
# Social Media Use and Loneliness in Europe: Does Age Matter?
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from scipy import stats

pd.set_option('display.float_format', '{:.4f}'.format)

# ------------------------------------------------------------------
# 4.1  CARICAMENTO
# ------------------------------------------------------------------
df = pd.read_csv('output/dataset/eu_ls_clean.csv')
df['fascia_eta'] = pd.Categorical(df['fascia_eta'],
                                   categories=['16–34', '35–54', '55–74', '75+'],
                                   ordered=True)
df['intensita_sm2'] = df['intensita_sm'] ** 2
print(f"Dataset: {df.shape[0]} righe · {df['paese'].nunique()} paesi")

# ------------------------------------------------------------------
# 4.2  DEFINIZIONE MODELLI
# ------------------------------------------------------------------
# Modello BASE: solo controlli (sesso, education, income, paese)
# Modello PIENO: controlli + intensita_sm + intensita_sm²
# Il confronto tra i due isola il contributo NETTO di intensita_sm

formula_base = 'score_UCLA ~ sesso + education + income + C(paese)'
formula_full = ('score_UCLA ~ intensita_sm + intensita_sm2 '
                '+ sesso + education + income + C(paese)')

# ------------------------------------------------------------------
# 4.3  CONFRONTO MODELLI — TUTTO IL CAMPIONE
# ------------------------------------------------------------------
m_base = smf.ols(formula_base, data=df).fit(cov_type='HC3')
m_full = smf.ols(formula_full, data=df).fit(cov_type='HC3')

r2_base = m_base.rsquared
r2_full = m_full.rsquared
delta_r2 = r2_full - r2_base

# F-test per il contributo incrementale di intensita_sm e intensita_sm²
# H₀ del F-test: i due coefficienti aggiunti sono entrambi zero
n   = int(m_full.nobs)
k_full = m_full.df_model
k_base = m_base.df_model
f_stat = ((r2_full - r2_base) / 2) / ((1 - r2_full) / (n - k_full - 1))
f_pval = 1 - stats.f.cdf(f_stat, 2, n - k_full - 1)

print("\n=== Confronto modelli — tutto il campione ===")
print(f"R² modello base (solo controlli) : {r2_base:.4f}")
print(f"R² modello pieno (+ intensita_sm): {r2_full:.4f}")
print(f"ΔR²                              : {delta_r2:.4f} ({delta_r2*100:.2f}%)")
print(f"F-test contributo intensita_sm   : F = {f_stat:.2f}  p = {f_pval:.4f}")

# ------------------------------------------------------------------
# 4.4  R² PARZIALE DI intensita_sm
# ------------------------------------------------------------------
# R² parziale = varianza spiegata da intensita_sm DOPO aver rimosso
# tutto ciò che già spiegano i controlli
r2_parziale = (r2_full - r2_base) / (1 - r2_base)

print(f"\nR² parziale di intensita_sm      : {r2_parziale:.4f} ({r2_parziale*100:.2f}%)")
print("→ Quota di varianza RESIDUA (dopo controlli) spiegata da intensita_sm")

# ------------------------------------------------------------------
# 4.5  CONFRONTO MODELLI — PER FASCIA D'ETÀ
# ------------------------------------------------------------------
fasce = ['16–34', '35–54', '55–74', '75+']
risultati_confronto = []

print("\n=== Confronto modelli per fascia d'età ===")
for fascia in fasce:
    sub = df[df['fascia_eta'] == fascia].copy()
    n_f = len(sub)

    mb = smf.ols(formula_base, data=sub).fit(cov_type='HC3')
    mf = smf.ols(formula_full, data=sub).fit(cov_type='HC3')

    r2_b  = mb.rsquared
    r2_f  = mf.rsquared
    dr2   = r2_f - r2_b
    r2_p  = (r2_f - r2_b) / (1 - r2_b)

    k_f   = mf.df_model
    f_s   = ((r2_f - r2_b) / 2) / ((1 - r2_f) / (n_f - k_f - 1))
    f_p   = 1 - stats.f.cdf(f_s, 2, n_f - k_f - 1)
    sig   = '***' if f_p<0.001 else ('**' if f_p<0.01 else ('*' if f_p<0.05 else 'n.s.'))

    risultati_confronto.append({
        'fascia'      : fascia,
        'N'           : n_f,
        'R²_base'     : round(r2_b, 4),
        'R²_pieno'    : round(r2_f, 4),
        'ΔR²'         : round(dr2, 4),
        'R²_parziale' : round(r2_p, 4),
        'F'           : round(f_s, 2),
        'p_value'     : round(f_p, 4),
        'sig'         : sig
    })

    print(f"\n  Fascia {fascia}  (N={n_f})")
    print(f"    R² base → pieno : {r2_b:.4f} → {r2_f:.4f}  (ΔR² = {dr2:.4f})")
    print(f"    R² parziale     : {r2_p:.4f} ({r2_p*100:.2f}%)")
    print(f"    F-test          : F = {f_s:.2f}  p = {f_p:.4f}  {sig}")

tab = pd.DataFrame(risultati_confronto)
tab.to_csv('output/dataset/step4_confronto_modelli.csv', index=False)
print("\nSalvato: output/dataset/step4_confronto_modelli.csv")

# ------------------------------------------------------------------
# 4.6  VERIFICA H₀ / H₁ — lettura dal confronto modelli
# ------------------------------------------------------------------
print("\n=== Verifica ipotesi (confronto modelli) ===")
print("H₀: intensita_sm non aggiunge potere esplicativo in nessuna fascia")
print("H₁: intensita_sm aggiunge potere esplicativo e in misura variabile per fascia\n")

for r in risultati_confronto:
    esito = ("→ contributo SIGNIFICATIVO (contro H₀)"
             if r['p_value'] < 0.05
             else "→ contributo NON significativo (a favore H₀)")
    print(f"  {r['fascia']:6s}  ΔR²={r['ΔR²']:.4f}  R²parz={r['R²_parziale']:.4f}"
          f"  p={r['p_value']:.4f} {r['sig']}  {esito}")

# ------------------------------------------------------------------
# 4.7  GRAFICI
# ------------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colori_sig = ['#4C72B0' if r['p_value'] < 0.05 else '#AAAAAA'
              for r in risultati_confronto]

# --- Fig A: ΔR² per fascia con significatività ---
delta_vals = [r['ΔR²'] for r in risultati_confronto]
bars = axes[0].bar(fasce, delta_vals, color=colori_sig, edgecolor='white', width=0.5)
axes[0].axhline(0, color='black', linewidth=0.8, linestyle='--')
axes[0].set_title('Contributo incrementale di intensita_sm (ΔR²) per fascia')
axes[0].set_xlabel("Fascia d'età")
axes[0].set_ylabel('ΔR² (R² pieno − R² base)')
for i, (bar, r) in enumerate(zip(bars, risultati_confronto)):
    axes[0].text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + 0.0003,
                 r['sig'], ha='center', va='bottom', fontsize=12)

# --- Fig B: R² parziale per fascia ---
rp_vals = [r['R²_parziale'] for r in risultati_confronto]
bars2 = axes[1].bar(fasce, rp_vals, color=colori_sig, edgecolor='white', width=0.5)
axes[1].set_title('R² parziale di intensita_sm per fascia d\'età')
axes[1].set_xlabel("Fascia d'età")
axes[1].set_ylabel('R² parziale')
for i, (bar, r) in enumerate(zip(bars2, risultati_confronto)):
    axes[1].text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + 0.0003,
                 r['sig'], ha='center', va='bottom', fontsize=12)

# Legenda manuale significatività
from matplotlib.patches import Patch
legenda = [Patch(color='#4C72B0', label='p < 0.05 (significativo)'),
           Patch(color='#AAAAAA', label='p ≥ 0.05 (non significativo)')]
axes[1].legend(handles=legenda, fontsize=9)

plt.tight_layout()
fig.savefig('output/figures/step4_confronto_modelli.png', dpi=150, bbox_inches='tight')
plt.show()
print("Salvato: output/figures/step4_confronto_modelli.png")

In [ ]:
# ============================================================
# STEP 4 — Random Forest (ML supervisionato)
# Does Social Media Use Reduce Loneliness?
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_absolute_error, mean_squared_error
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.float_format', '{:.4f}'.format)

# ------------------------------------------------------------------
# 4.1  CARICAMENTO E PREPARAZIONE FEATURE
# ------------------------------------------------------------------
df = pd.read_csv('output/dataset/eu_ls_clean.csv')
df['fascia_eta'] = pd.Categorical(df['fascia_eta'],
                                   categories=['16–34', '35–54', '55–74', '75+'],
                                   ordered=True)

# Termine quadratico — stesso dell'OLS
df['intensita_sm2'] = df['intensita_sm'] ** 2

# Dummies paese — stesso approccio dell'OLS (drop_first=True, paese 1 = riferimento)
paese_dummies = pd.get_dummies(df['paese'], prefix='paese', drop_first=True)

FEATURES  = ['intensita_sm', 'intensita_sm2', 'sesso', 'education', 'income']
X_base    = pd.concat([df[FEATURES], paese_dummies], axis=1).astype(float)
y         = df['score_UCLA']

fasce  = ['16–34', '35–54', '55–74', '75+']
colori = {'16–34': '#4C72B0', '35–54': '#DD8452',
          '55–74': '#55A868', '75+':   '#C44E52'}

print(f"Dataset: {df.shape[0]} righe · {X_base.shape[1]} feature totali")
print(f"Feature individuali: {FEATURES}")
print(f"Dummies paese: {paese_dummies.shape[1]}")

# ------------------------------------------------------------------
# 4.2  MODELLO RF — TUTTO IL CAMPIONE
# ------------------------------------------------------------------
print("\n=== RF — tutto il campione ===")

X_tr_full, X_te_full, y_tr_full, y_te_full = train_test_split(
    X_base, y, test_size=0.20, random_state=42)

rf_full = RandomForestRegressor(
    n_estimators=200, max_features='sqrt',
    min_samples_leaf=20, random_state=42, n_jobs=-1)
rf_full.fit(X_tr_full, y_tr_full)

y_pred_full = rf_full.predict(X_te_full)
mae_full    = mean_absolute_error(y_te_full, y_pred_full)
rmse_full   = np.sqrt(mean_squared_error(y_te_full, y_pred_full))

print(f"  MAE  = {mae_full:.4f}")
print(f"  RMSE = {rmse_full:.4f}")

# ------------------------------------------------------------------
# 4.3  MODELLO RF — SEPARATO PER FASCIA D'ETÀ
# ------------------------------------------------------------------
print("\n=== RF — per fascia d'età ===")

modelli  = {}   # {fascia: (rf, X_tr, X_te, y_tr, y_te)}
metriche = []
perm_imp = {}   # {fascia: Series importances_mean}

for fascia in fasce:
    mask  = df['fascia_eta'] == fascia
    X_f   = X_base[mask].reset_index(drop=True)
    y_f   = y[mask].reset_index(drop=True)

    X_tr, X_te, y_tr, y_te = train_test_split(
        X_f, y_f, test_size=0.20, random_state=42)

    rf = RandomForestRegressor(
        n_estimators=200, max_features='sqrt',
        min_samples_leaf=20, random_state=42, n_jobs=-1)
    rf.fit(X_tr, y_tr)
    modelli[fascia] = (rf, X_tr, X_te, y_tr, y_te)

    y_pred = rf.predict(X_te)
    mae    = mean_absolute_error(y_te, y_pred)
    rmse   = np.sqrt(mean_squared_error(y_te, y_pred))

    metriche.append({'Fascia': fascia, 'N_train': len(X_tr),
                     'N_test':  len(X_te),
                     'MAE':     round(mae, 4),
                     'RMSE':    round(rmse, 4)})

    # Permutation importance (preferita a impurity — non distorta)
    perm = permutation_importance(
        rf, X_te, y_te, n_repeats=10, random_state=42, n_jobs=-1)
    perm_imp[fascia] = pd.Series(
        perm.importances_mean, index=X_base.columns)

    top5 = perm_imp[fascia].nlargest(5)
    print(f"\n  Fascia {fascia}  (N_train={len(X_tr)}, N_test={len(X_te)})")
    print(f"    MAE = {mae:.4f}  |  RMSE = {rmse:.4f}")
    print("    Top 5 permutation importance:")
    for feat, imp in top5.items():
        print(f"      {feat:22s}  {imp:.4f}")

# ------------------------------------------------------------------
# 4.4  TABELLA METRICHE (Tab. 2)
# ------------------------------------------------------------------
tab_metriche = pd.DataFrame(metriche)
print("\n=== Tab. 2 — MAE e RMSE per fascia ===")
print(tab_metriche.to_string(index=False))
tab_metriche.to_csv('output/dataset/step4_metriche.csv', index=False)
print("\nSalvato: output/dataset/step4_metriche.csv")

# ------------------------------------------------------------------
# 4.5  CONFRONTO OLS × RF (Tab. comparativa)
# ------------------------------------------------------------------
ols_tab = pd.read_csv('output/dataset/step3_coefficienti.csv')
ols_sm  = ols_tab[ols_tab['variabile'].isin(['intensita_sm', 'intensita_sm2'])]

print("\n=== Tab. comparativa — OLS vs RF per fascia ===")
righe_conf = []
for fascia in fasce:
    sub  = ols_sm[ols_sm['fascia'] == fascia]
    sig1 = sub[sub['variabile'] == 'intensita_sm']['sig'].values[0]
    sig2 = sub[sub['variabile'] == 'intensita_sm2']['sig'].values[0]

    imp_sorted = perm_imp[fascia].sort_values(ascending=False)
    nomi       = list(imp_sorted.index)
    rank_sm    = nomi.index('intensita_sm')  + 1
    rank_sm2   = nomi.index('intensita_sm2') + 1

    # Accordo: entrambi i metodi concordano sull'importanza di intensita_sm?
    accordo = "✓" if (sig1 != 'n.s.' and rank_sm <= 5) else (
              "~" if (sig2 != 'n.s.' and rank_sm2 <= 5) else "✗")

    righe_conf.append({
        'Fascia':           fascia,
        'OLS β₁ sig':       sig1,
        'OLS β₂ sig':       sig2,
        'RF rank sm':       rank_sm,
        'RF rank sm²':      rank_sm2,
        'Accordo OLS×RF':   accordo
    })
    print(f"  {fascia:6s}  OLS β₁={sig1:5s}  β₂={sig2:5s} | "
          f"RF rank sm=#{rank_sm}  sm²=#{rank_sm2}  accordo={accordo}")

tab_conf = pd.DataFrame(righe_conf)
tab_conf.to_csv('output/dataset/step4_confronto_ols_rf.csv', index=False)
print("\nSalvato: output/dataset/step4_confronto_ols_rf.csv")

# ------------------------------------------------------------------
# 4.6  GRAFICI — step4_rf.png
# ------------------------------------------------------------------
fig = plt.figure(figsize=(18, 14))
fig.suptitle("Step 4 — Random Forest: Feature Importance, PDP e Confronto OLS",
             fontsize=13, fontweight='bold')

# ── Fila 1: Permutation importance (top 5) per fascia (Fig. 6) ──
for idx, fascia in enumerate(fasce):
    ax = fig.add_subplot(3, 4, idx + 1)
    top5 = perm_imp[fascia].nlargest(5).sort_values()

    bar_colors = ['#C44E52' if 'intensita_sm' in feat else '#AEC6CF'
                  for feat in top5.index]
    ax.barh(top5.index, top5.values, color=bar_colors, edgecolor='white')
    ax.axvline(0, color='black', linewidth=0.6)
    ax.set_title(f'Fascia {fascia}', fontsize=10, fontweight='bold')
    ax.set_xlabel('Permutation importance', fontsize=8)
    ax.tick_params(axis='y', labelsize=8)

# ── Fila 2 sinistra: PDP manuale — intensita_sm × fascia ──
ax_pdp = fig.add_subplot(3, 2, 3)
for fascia in fasce:
    rf, X_tr, X_te, y_tr, y_te = modelli[fascia]
    medie_pdp = []
    for val in range(9):
        X_tmp = X_te.copy()
        X_tmp['intensita_sm']  = val
        X_tmp['intensita_sm2'] = val ** 2
        medie_pdp.append(rf.predict(X_tmp).mean())
    ax_pdp.plot(range(9), medie_pdp, marker='o', linewidth=2,
                label=fascia, color=colori[fascia])

ax_pdp.axhline(df['score_UCLA'].mean(), color='gray',
               linestyle='--', linewidth=0.8, label='Media globale')
ax_pdp.set_title('Partial Dependence Plot — intensita_sm per fascia (RF)')
ax_pdp.set_xlabel('Livello intensita_sm (0 = non utente, 8 = uso massimo)')
ax_pdp.set_ylabel('Score UCLA predetto (RF)')
ax_pdp.set_xticks(range(9))
ax_pdp.legend(title="Fascia d'età", fontsize=8)

# ── Fila 2 destra: Confronto PDP (RF) vs curva fitted OLS — fascia 16–34 ──
ax_conf = fig.add_subplot(3, 2, 4)

# PDP RF per 16-34
rf_1634, _, X_te_1634, _, _ = modelli['16–34']
medie_rf = []
for val in range(9):
    X_tmp = X_te_1634.copy()
    X_tmp['intensita_sm']  = val
    X_tmp['intensita_sm2'] = val ** 2
    medie_rf.append(rf_1634.predict(X_tmp).mean())

# Curva OLS per 16-34 (β₁ e β₂ dalla tabella coefficienti)
b1 = ols_sm.loc[(ols_sm['fascia'] == '16–34') &
                (ols_sm['variabile'] == 'intensita_sm'),  'beta'].values[0]
b2 = ols_sm.loc[(ols_sm['fascia'] == '16–34') &
                (ols_sm['variabile'] == 'intensita_sm2'), 'beta'].values[0]

sm_grid  = np.linspace(0, 8, 100)
y_ols_tr = b1 * sm_grid + b2 * sm_grid**2
# Centro entrambe le serie sulla loro media per sovrapporle leggibilmente
ols_centered = y_ols_tr - np.mean(b1 * np.arange(9) + b2 * np.arange(9)**2)
rf_centered  = np.array(medie_rf) - np.mean(medie_rf)

ax_conf.plot(sm_grid, ols_centered, linewidth=2, linestyle='--',
             label='OLS — curva fitted', color='#DD8452')
ax_conf.plot(range(9), rf_centered, marker='o', linewidth=2,
             label='RF — PDP', color='#4C72B0')
ax_conf.axhline(0, color='gray', linestyle='--', linewidth=0.6)
ax_conf.set_title("Confronto forma relazione — fascia 16–34\n(serie centrate sulla media)")
ax_conf.set_xlabel('Livello intensita_sm')
ax_conf.set_ylabel('Deviazione dalla media (score UCLA)')
ax_conf.set_xticks(range(9))
ax_conf.legend(fontsize=9)

# ── Fila 3: Permutation importance aggregata (tutto il campione) ──
ax_all = fig.add_subplot(3, 1, 3)
perm_full = permutation_importance(
    rf_full, X_te_full, y_te_full, n_repeats=10, random_state=42, n_jobs=-1)
imp_full = pd.Series(perm_full.importances_mean, index=X_base.columns)

# Seleziona solo feature non-paese per leggibilità + top 3 paese
imp_individuale = imp_full[FEATURES].sort_values()
imp_paese_top3  = imp_full[[c for c in imp_full.index
                             if c.startswith('paese_')]].nlargest(3).sort_values()
imp_plot        = pd.concat([imp_paese_top3, imp_individuale])

bar_colors_all = ['#C44E52' if 'intensita_sm' in feat
                  else ('#4C72B0' if not feat.startswith('paese_') else '#AEC6CF')
                  for feat in imp_plot.index]

ax_all.barh(imp_plot.index, imp_plot.values,
            color=bar_colors_all, edgecolor='white')
ax_all.axvline(0, color='black', linewidth=0.6)
ax_all.set_title('Permutation importance — tutto il campione'
                 ' (rosso = intensita_sm, blu = controlli individuali, grigio = paese)')
ax_all.set_xlabel('Permutation importance (mean decrease in R²)')
ax_all.tick_params(axis='y', labelsize=9)

plt.tight_layout()
fig.savefig('output/figures/step4_rf.png', dpi=150, bbox_inches='tight')
plt.show()
print("\nSalvato: output/figures/step4_rf.png")

# ------------------------------------------------------------------
# 4.7  SHAP — fascia 16–34 (la più informativa)
# ------------------------------------------------------------------
try:
    import shap

    rf_1634, _, X_te_1634, _, _ = modelli['16–34']
    shap_sample  = X_te_1634.sample(min(500, len(X_te_1634)), random_state=42)
    explainer    = shap.TreeExplainer(rf_1634)
    shap_vals    = explainer.shap_values(shap_sample)

    fig_shap, axes_shap = plt.subplots(1, 2, figsize=(14, 5))

    # Bar SHAP — importanza media |SHAP| per feature (solo feature individuali + top paese)
    shap_mean = pd.Series(np.abs(shap_vals).mean(axis=0), index=X_base.columns)
    shap_ind  = shap_mean[FEATURES].sort_values()
    shap_top3p = shap_mean[[c for c in shap_mean.index
                             if c.startswith('paese_')]].nlargest(3).sort_values()
    shap_plot = pd.concat([shap_top3p, shap_ind])

    bar_colors_shap = ['#C44E52' if 'intensita_sm' in f
                       else ('#4C72B0' if not f.startswith('paese_') else '#AEC6CF')
                       for f in shap_plot.index]
    axes_shap[0].barh(shap_plot.index, shap_plot.values,
                      color=bar_colors_shap, edgecolor='white')
    axes_shap[0].set_title('SHAP mean |value| — fascia 16–34')
    axes_shap[0].set_xlabel('SHAP mean |value|')
    axes_shap[0].tick_params(axis='y', labelsize=9)

    # Scatter SHAP — intensita_sm: valore SHAP vs livello di intensità
    sm_idx  = list(X_base.columns).index('intensita_sm')
    sm_vals = shap_sample['intensita_sm'].values
    sm_shap = shap_vals[:, sm_idx]

    axes_shap[1].scatter(sm_vals, sm_shap, alpha=0.3, s=15,
                         color='#4C72B0', edgecolors='none')
    axes_shap[1].axhline(0, color='black', linewidth=0.8, linestyle='--')
    # Trend SHAP medio per livello
    for v in range(9):
        mask_v = sm_vals == v
        if mask_v.sum() > 0:
            axes_shap[1].scatter(v, sm_shap[mask_v].mean(),
                                 color='#C44E52', s=60, zorder=5)
    axes_shap[1].set_title('SHAP per intensita_sm — fascia 16–34\n'
                            '(punto rosso = media per livello)')
    axes_shap[1].set_xlabel('Livello intensita_sm')
    axes_shap[1].set_ylabel('Valore SHAP (contributo a score UCLA)')
    axes_shap[1].set_xticks(range(9))

    fig_shap.suptitle('Step 4 — SHAP Analysis (fascia 16–34)',
                       fontsize=12, fontweight='bold')
    plt.tight_layout()
    fig_shap.savefig('output/figures/step4_shap.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Salvato: output/figures/step4_shap.png")

except ImportError:
    print("\n[INFO] shap non installato — esegui: pip install shap")
    print("       I grafici SHAP verranno saltati.")

# ------------------------------------------------------------------
# 4.8  RIEPILOGO FINALE
# ------------------------------------------------------------------
print("\n" + "="*60)
print("RIEPILOGO STEP 4")
print("="*60)
print(f"\nRF tutto il campione — MAE={mae_full:.4f}  RMSE={rmse_full:.4f}")
print("\nR² basso atteso (come OLS): la solitudine è influenzata da fattori")
print("non osservabili (personalità, storia, eventi di vita).")
print("\nFunzione del RF in questo progetto:")
print("  → Valida la struttura dell'effetto (importanza relativa delle feature)")
print("  → Cattura la forma a U senza imporla (PDP vs curva OLS: accordo?)")
print("  → Triangolazione metodologica classico+ML")